# import Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_url = "http://s3.amazonaws.com/assets.datacamp.com/course/Kaggle/train.csv" 
train = pd.read_csv(train_url)
test_url = "http://s3.amazonaws.com/assets.datacamp.com/course/Kaggle/test.csv" 
test = pd.read_csv(test_url) 

# Data preprocessing

In [3]:
def standardize(X):
    """
    Standardize features to have mean=0 and std=1
    Formula: X_scaled = (X - mean) / std
    """
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)

    # ป้องกันการหารด้วย 0
    std[std == 0] = 1

    X_scaled = (X - mean) / std
    return X_scaled, mean, std

In [4]:
def preprocess(df):
    df.loc[df["Embarked"] == "S", "Embarked"] = 0
    df.loc[df["Embarked"] == "C", "Embarked"] = 1
    df.loc[df["Embarked"] == "Q", "Embarked"] = 2
    df.loc[df["Sex"] == "male", "Sex"] = 0
    df.loc[df["Sex"] == "female", "Sex"] = 1
    df["Sex"] = df["Sex"].astype(int)
    return df

In [5]:
train_pre = preprocess(train)
test_pre = preprocess(test)

age_median = train_pre["Age"].median()
embarked_mode = train_pre["Embarked"].mode().iloc[0]
print("Age median:", age_median)
print("Embarked mode:", embarked_mode)

train_pre["Age"] = train_pre["Age"].fillna(age_median)
train_pre["Embarked"] = train_pre["Embarked"].fillna(embarked_mode)
data = np.array(train_pre[["Pclass","Sex","Age","Embarked"]].values, dtype = float)

test_pre["Age"] = test_pre["Age"].fillna(age_median)
test_pre["Embarked"] = test_pre["Embarked"].fillna(embarked_mode)
test_data = np.array(test_pre[["Pclass", "Sex", "Age", "Embarked"]].values, dtype=float)

y = np.array(train["Survived"].values, dtype=float)

Age median: 28.0
Embarked mode: 0


/tmp/ipykernel_45856/1591505398.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_pre["Embarked"] = train_pre["Embarked"].fillna(embarked_mode)
/tmp/ipykernel_45856/1591505398.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_pre["Embarked"] = test_pre["Embarked"].fillna(embarked_mode)


# Train & Test Function

In [6]:
def calculate_mse(y_predict, y):
    m = len(y)
    mse = (1/m) * np.sum((y_predict - y)**2)
    return mse

def calculate_accuracy(y_predict, y):
    predictions_binary = (y_predict >= 0.5).astype(int)
    accuracy = np.mean(predictions_binary == y)
    return accuracy

In [7]:
def linear_regression_gradient_descent(X, y, learning_rate=0.01, iterations=1000):
    m = len(y)
    X = np.c_[np.ones(m), X]

    n_features = X.shape[1]
    theta = np.zeros(n_features)

    for i in range(iterations):
        h = X.dot(theta)
        gradient = (1/m) * X.T.dot(h - y)
        theta = theta - learning_rate * gradient

    return theta

In [8]:
def predict(X, theta):
    m = X.shape[0]
    X = np.c_[np.ones(m), X]
    predictions = X.dot(theta)
    return predictions

def convert_to_binary(y):
    return (y >= 0.5).astype(int)

# T10

In [9]:
data_scaled, train_mean, train_std = standardize(data)
theta = linear_regression_gradient_descent(data_scaled, y, learning_rate=0.001, iterations=100000)
y_predict = predict(data_scaled, theta)
train_mse = calculate_mse(y_predict, y)
train_accuracy = calculate_accuracy(convert_to_binary(y_predict), y)
print("theta:", theta)
print(f"Training MSE: {train_mse}")
print(f"Training Accuracy: {train_accuracy}")

theta: [ 0.38383838 -0.15746036  0.2344979  -0.06576928  0.0312026 ]
Training MSE: 0.14492573766134814
Training Accuracy: 0.7991021324354658


In [10]:
test_data_scaled = (test_data - train_mean) / train_std
predictions = predict(test_data_scaled, theta)

In [11]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': convert_to_binary(predictions)
})
submission.to_csv('T10_submission.csv', index=False)

# T12

In [12]:
def preprocess_high_order(data_scaled):
    data_high_order = np.array(data_scaled)
    x4 = data_high_order[:, 2] ** 2
    x5 = data_high_order[:, 1] * data_high_order[:, 2]
    x6 = data_high_order[:, 1] * data_high_order[:, 0]
    x7 = data_high_order[:, 1] * data_high_order[:, 3]
    x8 = data_high_order[:, 2] * data_high_order[:, 0]
    x9 = data_high_order[:, 2] * data_high_order[:, 3]
    data_high_order = np.c_[data_high_order, x4, x5, x6, x7, x8, x9]
    return data_high_order

In [13]:
data_high_order = preprocess_high_order(data)
data_high_order, train_mean_high_order, train_std_high_order = standardize(data_high_order)
theta_high_order = linear_regression_gradient_descent(data_high_order, y, learning_rate=0.1, iterations=1000)
y_predict_high_order = predict(data_high_order, theta_high_order)
train_high_order_mse = calculate_mse(y_predict_high_order, y)
train_high_order_accuracy = calculate_accuracy(convert_to_binary(y_predict_high_order), y)
print("theta:", theta_high_order)
print(f"Training MSE: {train_high_order_mse}")
print(f"Training Accuracy: {train_high_order_accuracy}")

theta: [ 0.38383838 -0.14127159  0.27975698 -0.22210426  0.04887783  0.12379878
  0.0790349  -0.1439937   0.0426937   0.03169695 -0.04106522]
Training MSE: 0.1389825890288719
Training Accuracy: 0.819304152637486


In [14]:
test_high_order = preprocess_high_order(test_data)
test_data_scaled = (test_high_order - train_mean_high_order) / train_std_high_order
predictions_high_order = predict(test_data_scaled, theta_high_order)

In [15]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': convert_to_binary(predictions_high_order)
})
submission.to_csv('T12_submission.csv', index=False)

# T13

In [16]:
data_low_feat = np.array(train_pre[["Sex","Age"]].values, dtype = float)
data_low_feat_scaled, train_low_feat_mean, train_low_feat_std = standardize(data_low_feat)
theta_low_feat = linear_regression_gradient_descent(data_low_feat_scaled, y, learning_rate=0.01, iterations=10000)
y_predict_low_feat = predict(data_low_feat_scaled, theta_low_feat)
train_mse_low_feat = calculate_mse(y_predict_low_feat, y)
train_accuracy_low_feat = calculate_accuracy(convert_to_binary(y_predict_low_feat), y)
print("theta:", theta_low_feat)
print(f"Training MSE: {train_mse_low_feat}")
print(f"Training Accuracy: {train_accuracy_low_feat}")

theta: [ 0.38383838  0.26341541 -0.01018773]
Training MSE: 0.16657939408141703
Training Accuracy: 0.7867564534231201


In [17]:
test_data_low_feat = np.array(test_pre[["Sex", "Age"]].values, dtype=float)
test_data_low_feat_scaled = (test_data_low_feat - train_low_feat_mean) / train_low_feat_std
predictions_low_feat = predict(test_data_low_feat_scaled, theta_low_feat)

In [18]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': convert_to_binary(predictions_low_feat)
})
submission.to_csv('T13_submission.csv', index=False)

# Matrix Inversion

In [19]:
m = len(y)
X = np.c_[np.ones(m), data_scaled]
matrix_inversion = np.linalg.inv(X.T.dot(X))
theta_normal = matrix_inversion.dot(X.T).dot(y)

In [20]:
for row in matrix_inversion.tolist():
    print(row)

[0.001122334455667789, 1.3336326605640432e-20, -2.44846666182638e-19, -4.362348113840282e-19, 2.1860174916300365e-19]
[1.3336326605640392e-20, 0.001312565320773622, 0.00022021794148560338, 0.0004632660556023183, -8.141142949442723e-05]
[-2.4484666618263807e-19, 0.00022021794148560336, 0.0011821914183460767, 0.00016946032841411423, -0.00014631861771918826]
[-4.362348113840283e-19, 0.00046326605560231826, 0.00016946032841411423, 0.001293285186081991, -2.907283520233438e-05]
[2.186017491630037e-19, -8.141142949442722e-05, -0.00014631861771918824, -2.9072835202334368e-05, 0.0011428449292687458]


In [21]:
print("Theta from Normal Equation:")
print(theta_normal)
print("Theta from Gradient Descent:")
print(theta)

Theta from Normal Equation:
[ 0.38383838 -0.15746036  0.2344979  -0.06576928  0.0312026 ]
Theta from Gradient Descent:
[ 0.38383838 -0.15746036  0.2344979  -0.06576928  0.0312026 ]


In [22]:
y_predict_matrix_inversion = predict(data_scaled, theta_normal)
train_mse_matrix_inversion = calculate_mse(y_predict_matrix_inversion, y)
train_accuracy_matrix_inversion = calculate_accuracy(convert_to_binary(y_predict_matrix_inversion), y)
print("theta normal:", theta_normal)
print(f"Training MSE: {train_mse_matrix_inversion}")
print(f"Training Accuracy: {train_accuracy_matrix_inversion}")

theta normal: [ 0.38383838 -0.15746036  0.2344979  -0.06576928  0.0312026 ]
Training MSE: 0.14492573766134814
Training Accuracy: 0.7991021324354658
